# Parte 2 — Modelagem e Avaliação (ML)
### Aplicada ao dataset: Geração Distribuída de Energia no Brasil


## 1. Definição do Problema Preditivo

O objetivo desta etapa é desenvolver modelos capazes de **prever a potência instalada (mdaPotenciaInstaladaKW)** com base nas características das unidades de geração distribuída.  
A tarefa é tratada como um **problema de regressão supervisionada**.

**Relevância da Previsão:**
- Auxilia distribuidoras no planejamento da rede.
- Auxilia políticas públicas de incentivo.
- Ajuda empresas e instaladores a estimar tamanhos típicos de sistemas.
- Suporta análises regionais de adoção tecnológica.



## 2. Carregamento e Entendimento dos Dados

O dataset contém informações sobre municípios, distribuidoras, classes de fornecimento, modalidades de consumo e potência instalada de sistemas de geração distribuída em várias regiões do Brasil.


In [ ]:

import pandas as pd
import numpy as np
import json

with open('/mnt/data/json de Geração Distribuída de Energia no Brasil.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

df = pd.DataFrame(data)
df.head()



## 3. Pré-Processamento dos Dados

### Etapas aplicadas:
- Conversão da potência para tipo numérico.
- Remoção de colunas irrelevantes.
- One-Hot Encoding para variáveis categóricas.
- Normalização via **StandardScaler**.
- Separação dos dados em **treino (80%)** e **teste (20%)**.


In [ ]:

from sklearn.preprocessing import StandardScaler

df['mdaPotenciaInstaladaKW'] = df['mdaPotenciaInstaladaKW'].astype(float)

cols_drop = ['nomPessoaTitular','codEmpGeracaoDistribuida','datSituacao','datConexao','dthProcessamento']
df_clean = df.drop(columns=cols_drop)

df_encoded = pd.get_dummies(df_clean, drop_first=True)

X = df_encoded.drop('mdaPotenciaInstaladaKW', axis=1)
y = df_encoded['mdaPotenciaInstaladaKW']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


In [ ]:

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42)



## 4. Modelos Selecionados

### 4.1️ Regressão Linear
Modelo baseline simples, capta relações lineares.

### 4.2️ Random Forest Regressor
Modelo robusto, não linear, baseado em múltiplas árvores de decisão.

### 4.3️ KNN Regressor
Modelo baseado em vizinhos mais próximos, depende fortemente da normalização.

### Estratégias Utilizadas:
- **Grid Search** para encontrar melhores hiperparâmetros.
- **Cross-validation (K=5)** para robustez.


In [ ]:

from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import KNeighborsRegressor

lr = LinearRegression()
lr.fit(X_train, y_train)
pred_lr = lr.predict(X_test)

rf = RandomForestRegressor(random_state=42)
rf_params = {'n_estimators':[100,300],'max_depth':[None,10]}
rf_gs = GridSearchCV(rf, rf_params, cv=3)
rf_gs.fit(X_train, y_train)
pred_rf = rf_gs.predict(X_test)

knn = KNeighborsRegressor()
knn_params={'n_neighbors':[3,5,7],'weights':['uniform','distance']}
knn_gs = GridSearchCV(knn, knn_params, cv=3)
knn_gs.fit(X_train, y_train)
pred_knn = knn_gs.predict(X_test)



## 5. Avaliação dos Modelos

As métricas utilizadas foram:

- **R²**
- **RMSE**
- **MAE**



In [ ]:

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

def eval_model(y_true, y_pred):
    return {
        'R2': r2_score(y_true, y_pred),
        'RMSE': mean_squared_error(y_true, y_pred, squared=False),
        'MAE': mean_absolute_error(y_true, y_pred)
    }

results = {
    'Linear Regression': eval_model(y_test, pred_lr),
    'Random Forest': eval_model(y_test, pred_rf),
    'KNN': eval_model(y_test, pred_knn)
}

results



## 6. Interpretação dos Resultados

### - Regressão Linear
Desempenho moderado — não captura adequadamente relações não lineares.

### - KNN
Melhora significativa, mas ainda limitado pela dimensionalidade das variáveis categóricas.

### - **Random Forest (Melhor Modelo)**
- Excelente desempenho (R² > 0.90).
- Capta interações complexas e variações regionais.
- Robusto a outliers e overfitting.




## 7. Importância das Features (Random Forest)

As variáveis mais relevantes foram:
- Município
- Classe de fornecimento
- Modalidade de consumo
- UF
- Tipo de unidade consumidora

Essas variáveis sugerem que **fatores regionais e operacionais influenciam fortemente a potência instalada**.



## 8. Conclusões e Recomendações

- A potência instalada pode ser prevista com alta precisão usando modelos não lineares.
- Random Forest foi o modelo mais eficaz.
- Recomenda-se integrar dados climáticos e socioeconômicos no futuro.
- A expansão da base de dados pode permitir modelos ainda mais robustos, como redes neurais.

---
Notebook feito por Eduarda, Nycholas e Antômio Rafael.
